# 02 - Nettoyage des données DPE

**Objectif** : préparer des données propres pour les départements 59, 69 et 75.
Chaque département est traité séparément, avec les mêmes règles appliquées à tous.

## 1. Chargement
Les trois fichiers `processed` (neufs + existants déjà fusionnés par département) sont chargés dans un dictionnaire : une entrée par département.

In [1]:
import sys
print(sys.executable)

c:\Users\albar\Documents\GitHub\SISE-CONSO-ANALYSIS\.venv\Scripts\python.exe


In [2]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocessing import DataPreprocessor

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 50)

RAW = "../data/raw/2026-10-12_132605/"
DEPARTEMENTS = ["59", "69", "75"]

datasets = {
    d: {"dpe_existants": pd.read_parquet(RAW + f"{d}/dpe_existants.parquet"),
        "dpe_neufs": pd.read_parquet(RAW + f"{d}/dpe_neufs.parquet")}
    for d in DEPARTEMENTS
}
# Fusion seule (224 colonnes, aucun nettoyage), comme l'ancien fichier processed
dfs = DataPreprocessor(appliquer_nettoyage=False).process(datasets)

for d, df in dfs.items():
    print(d, df.shape)

59 (140344, 226)
69 (144964, 224)
75 (108644, 229)


## 2. Comparaison des trois départements
Paris compte moins de lignes et de colonnes que les deux autres départements. On vérifie la répartition neufs / existants et les colonnes manquantes.

In [ ]:
# Répartition neufs / existants par département
for d, df in dfs.items():
    print(d, df["origine"].value_counts().to_dict())

# Colonnes qui ne sont pas présentes dans tous les départements
toutes = set().union(*[set(df.columns) for df in dfs.values()])
for d, df in dfs.items():
    absentes = sorted(toutes - set(df.columns))
    print(f"\n{d} : {len(absentes)} colonne(s) absente(s) -> {absentes}")

**Observation (étape 2)**

- **Paris** a 8 644 logements neufs, contre 10 000 pour Lille et Lyon. Pour Paris, on a donc sans doute tous les DPE neufs.
- Dans nos fichiers, il y a environ **moitié neufs, moitié existants**. Ce n'est pas la réalité : c'est parce qu'on a récupéré au maximum 10 000 lignes de chaque type. Il ne faut donc pas en tirer de conclusion sur le département.
- Paris n'a pas 3 colonnes très rares (sur le solaire et un numéro RPLS). Elles ne servent pas dans le projet, ce n'est pas un problème.

## 3. Vue d'ensemble par département

Pour chaque département, on regarde : les doublons, la période couverte et la répartition des étiquettes DPE.

In [ ]:
for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print("Doublons sur numero_dpe :", df["numero_dpe"].duplicated().sum())

    dates = pd.to_datetime(df["date_reception_dpe"], errors="coerce")
    print("Période :", dates.min().date(), "->", dates.max().date())

    print("Étiquettes :")
    print(pd.crosstab(df["origine"], df["etiquette_dpe"]))
    print()

**Observations (étape 3)**
- Aucun doublon dans les trois départements.
- Les logements neufs sont presque tous en A, B ou C.
- Les logements existants de Paris sont plus souvent en E que ceux de Lille et de Lyon.
- Les neufs en G sont presque tous à Lyon (12), très peu à Lille (1) et aucun à Paris.
- Lille et Lyon s'arrêtent au 09/02/2026, Paris va jusqu'au 14/09/2026 : à vérifier, nos données ne couvrent peut-être pas toute la période.

## 4. Répartition des DPE dans le temps
On regarde combien de DPE il y a par année et par type, pour voir si l'échantillon couvre bien toute la période.

In [ ]:
for d, df in dfs.items():
    dates = pd.to_datetime(df["date_reception_dpe"], errors="coerce")
    print(f"===== Département {d} =====")
    print(pd.crosstab(dates.dt.year, df["origine"]))
    print("Dernière date par type :")
    print(dates.groupby(df["origine"]).max().dt.date)
    print()

**Observations (étape 4)**
- Existants de Lille et Lyon : environ la moitié des DPE sont de 2021, l'autre moitié de 2026, et presque aucun entre les deux.
- Existants de Paris : presque tous de 2026.
- Lille et Lyon s'arrêtent au 09/02/2026 : il manque les mois suivants.
- Conséquence : l'échantillon n'est pas représentatif dans le temps. On ne peut pas étudier l'évolution dans le temps avec ces données.

## 5. Les DPE de 2021 et de 2026 sont-ils comparables ?
Les règles de calcul du DPE ont pu changer entre les deux périodes. On compare, pour les logements existants, la répartition des étiquettes et la consommation médiane par année.

In [ ]:
for d, df in dfs.items():
    ex = df[df["origine"] == "dpe_existants"].copy()
    ex["annee"] = pd.to_datetime(ex["date_reception_dpe"], errors="coerce").dt.year
    print(f"===== Département {d} (existants) =====")

    # Part de chaque étiquette par année
    print(pd.crosstab(ex["annee"], ex["etiquette_dpe"], normalize="index").round(2))

    # Consommation médiane par année, pour les 3 énergies de chauffage les plus fréquentes
    top = ex["type_energie_principale_chauffage"].value_counts().head(3).index
    print(ex[ex["type_energie_principale_chauffage"].isin(top)]
          .pivot_table(index="annee", columns="type_energie_principale_chauffage",
                       values="conso_5_usages_par_m2_ep", aggfunc="median").round(0))
    print()

**Observations (étape 5)**
- Pour les logements existants, les étiquettes et la consommation médiane de 2021 et 2026 sont comparables dans chaque département : on garde toutes les années.
- Les années 2022 à 2025 ont trop peu de lignes (moins de 400) pour être interprétées.
- Paris est différent de Lille et de Lyon : plus de logements en E, F et G, et consommation électrique plus élevée. C'est une vraie différence du parc, pas une erreur.
- À mentionner dans le rapport : nos données sont concentrées sur 2021 et 2026, elles ne permettent pas d'étudier l'évolution dans le temps car cela dessinerait deux pics sur 2021 et 2026.

## 6. Consommation et valeurs aberrantes
Variable : `conso_5_usages_par_m2_ep` (kWh/m²/an). On cherche les valeurs impossibles ou extrêmes pour fixer un seuil, en regardant chaque département.

In [ ]:
col = "conso_5_usages_par_m2_ep"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[col].quantile([0.01, 0.5, 0.9, 0.99, 0.999, 1]).round(1))
    print("Valeurs <= 0  :", (df[col] <= 0).sum())
    print("Valeurs > 800 :", (df[col] > 800).sum())
    print("5 plus fortes valeurs :")
    print(df.nlargest(5, col)[["etiquette_dpe", col, "surface_habitable_logement", "origine"]])
    print()

**Observations (étape 6)**
- Aucune consommation nulle ou négative.
- Peu de valeurs au-dessus de 800 kWh/m²/an : 17 à Lille, 14 à Lyon, 42 à Paris (environ 0,12 % des lignes).
- Paris a des consommations plus élevées (médiane de 148 contre environ 95 ailleurs), ce qui est cohérent avec son parc ancien et ses petites surfaces.
- Deux cas parmi les valeurs extrêmes : des très petites surfaces (8 à 20 m²), où une forte consommation par m² est possible, et des surfaces normales (environ 80 m²) avec plus de 1 800 kWh/m²/an, très probablement des erreurs de saisie.

## 7. Choix du seuil maximum de consommation
On compare plusieurs seuils : combien de lignes supprimerait chacun, et à quelles surfaces correspondent les valeurs extrêmes.

In [ ]:
col = "conso_5_usages_par_m2_ep"
surf = "surface_habitable_logement"

# 1. Combien de lignes seraient supprimées selon le seuil ?
lignes = []
for d, df in dfs.items():
    for seuil in [600, 800, 1000, 1200]:
        n = (df[col] > seuil).sum()
        lignes.append({"departement": d, "seuil": seuil,
                       "lignes_supprimees": n,
                       "part_%": round(n / len(df) * 100, 2)})
pd.DataFrame(lignes).pivot(index="seuil", columns="departement",
                           values=["lignes_supprimees", "part_%"])

In [ ]:
# 2. À quelles surfaces correspondent les valeurs > 800 ?
for d, df in dfs.items():
    extremes = df[df[col] > 800]
    print(f"===== Département {d} : {len(extremes)} valeurs > 800 =====")
    print(pd.cut(extremes[surf], [0, 9, 20, 50, 1000]).value_counts().sort_index())
    print("Surface manquante :", extremes[surf].isna().sum())
    print()

**Observations (étape 7)**
- Un seuil à 600 kWh/m²/an est trop sévère : il supprimerait de vraies passoires thermiques (la classe G commence vers 420 kWh/m²/an).
- Au-dessus de 1 000, il ne reste que quelques valeurs isolées (8 à Lille, 3 à Lyon, 7 à Paris), très probablement des erreurs de saisie.
- À Paris, presque toutes les consommations supérieures à 800 concernent des logements de moins de 20 m², plausibles pour de très petits logements.
- À Lille et à Lyon, quelques valeurs supérieures à 800 concernent des logements de plus de 50 m² : peu crédibles.

**Décision proposée** : seuil maximum de consommation de **1 000 kWh/m²/an**. Il supprime 18 lignes sur 58 644 (0,03 %) et conserve les petits logements parisiens plausibles.

## 8. Surface et type de bâtiment
La règle sur la surface est liée à celle sur la consommation. On regarde les surfaces trop petites ou trop grandes, les surfaces manquantes, et le type de bâtiment concerné.

In [ ]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[surf].describe().round(1))
    print("Surface manquante :", df[surf].isna().sum())
    print("Surface < 9 m²    :", (df[surf] < 9).sum())
    print("Surface > 300 m²  :", (df[surf] > 300).sum())
    print("Surface > 500 m²  :", (df[surf] > 500).sum())
    print("Hauteur sous plafond :")
    print(df["hauteur_sous_plafond"].describe().round(2))
    print("Type de bâtiment, quand la surface est manquante :")
    print(df.loc[df[surf].isna(), "type_batiment"].value_counts())
    print()

**Observations (étape 8)**
- Des surfaces impossibles : maximum de 10 078 m² (Lille), 6 317 m² (Lyon) et 101 036 m² (Paris). Environ 100 à 125 lignes par département dépassent 500 m².
- Les surfaces manquantes sont presque toutes des DPE de type `immeuble` (410 sur 412 à Paris). Un DPE d'immeuble décrit un bâtiment entier, pas un logement.
- La hauteur sous plafond est parfois saisie en centimètres au lieu de mètres (maximum de 310 à Lille et 270 à Lyon, alors que la médiane est de 2,5).
- Paris compte 90 logements de moins de 9 m² (contre 2 à Lille et 3 à Lyon) : probablement de vraies chambres de service, mais le minimum de 3,4 m² est suspect.

## 9. Vérifications avant de fixer les règles
On regarde si les très grandes surfaces sont des immeubles, comment se répartissent les petites surfaces, et combien de hauteurs sous plafond sont en centimètres.

In [ ]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    print(f"===== Département {d} =====")

    # Surface par type de bâtiment
    print(df.groupby("type_batiment")[surf].agg(["count", "median", "max"]).round(1))

    # Type de bâtiment des surfaces > 300 m²
    print("Surfaces > 300 m², par type :", df.loc[df[surf] > 300, "type_batiment"].value_counts().to_dict())

    # Répartition des petites surfaces
    print("Petites surfaces :", pd.cut(df[surf], [0, 5, 7, 9]).value_counts().sort_index().to_dict())
    print()

**Observations (étape 9)**
- Les immeubles ont une surface médiane d'environ 1 000 m² (contre 40 à 110 m² pour les appartements et les maisons). Ils expliquent la plupart des surfaces supérieures à 300 m² et presque toutes les surfaces manquantes.
- Il reste quelques erreurs de saisie parmi les logements (par exemple un appartement de 3 932 m² à Paris).
- La hauteur sous plafond est parfois saisie en centimètres (270, 310) ou multipliée par 10 (25, 26). Il y a 119 valeurs supérieures à 5 m sur 58 644 lignes.
- Pour savoir si ces hauteurs concernent surtout les immeubles, on a comparé avant et après exclusion des immeubles (résultat ci-dessous).

**Décisions provisoires**
- Exclure les DPE de type `immeuble` (1 271 lignes, soit 2,2 %) : l'application concerne des logements, pas des bâtiments entiers.
- Supprimer les lignes sans surface.
- Surface minimale : 9 m². Surface maximale : à fixer (étape 10).
- Consommation maximale : 1 000 kWh/m²/an.

In [ ]:
h = "hauteur_sous_plafond"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print("Hauteur < 1.8 m       :", (df[h] < 1.8).sum())
    print("Hauteur entre 1.8 et 5 :", df[h].between(1.8, 5).sum())
    print("Hauteur > 5           :", (df[h] > 5).sum())
    print("Valeurs > 5 (extrait) :", sorted(df.loc[df[h] > 5, h].unique())[:15])
    print()

In [ ]:
h = "hauteur_sous_plafond"

for d, df in dfs.items():
    hautes = df[df[h] > 5]
    print(f"===== Département {d} : {len(hautes)} valeurs > 5 m =====")
    print(hautes["type_batiment"].value_counts().to_dict())

    sans_immeuble = df[df["type_batiment"] != "immeuble"]
    print("Hors immeubles :", (sans_immeuble[h] > 5).sum(), "valeurs > 5 m")
    print()

**Résultat (hauteur sous plafond)**
- Sur 119 valeurs supérieures à 5 m, seulement 4 concernent des immeubles : 115 concernent des appartements et des maisons (85 à Lille, 27 à Lyon, 3 à Paris).
- Exclure les immeubles ne suffit donc pas pour corriger cette variable.

**Décision (hauteur sous plafond)** : les valeurs hors de 1,8 à 5 m sont remplacées par « manquant ». On ne supprime pas la ligne et on ne devine pas l'unité de saisie (mètres ou centimètres). Le modèle remplacera le manquant par une valeur typique (la médiane) au moment de l'entraînement.

## 10. Surface maximale et coordonnées
On regarde la répartition des grandes surfaces parmi les logements (hors immeubles), puis le format des coordonnées pour la carte.

In [ ]:
surf = "surface_habitable_logement"

for d, df in dfs.items():
    logements = df[df["type_batiment"] != "immeuble"]
    print(f"===== Département {d} : {len(logements)} logements (hors immeubles) =====")
    print(pd.cut(logements[surf], [0, 9, 200, 300, 400, 500, 1000, 1e6]).value_counts().sort_index())
    print()

In [ ]:
for d, df in dfs.items():
    print(d, "-> coordonnées manquantes :", df["_geopoint"].isna().sum())

print()
print(dfs["69"]["_geopoint"].head())
print(type(dfs["69"]["_geopoint"].iloc[0]))

**Observations (étape 10)**
- Hors immeubles, plus de 97 % des logements ont une surface entre 9 et 200 m².
- De 200 à 500 m², ce sont de grandes maisons plausibles (environ 280 lignes au total).
- Au-dessus de 500 m², il ne reste que 17 lignes sur environ 57 000 (8 à Lille, 3 à Lyon, 6 à Paris) : erreurs de saisie probables.
- Paris compte 106 logements de 9 m² ou moins (contre 2 à Lille et 4 à Lyon).
- La colonne `_geopoint` est un texte au format « latitude,longitude », sans valeur manquante.

**Décisions (surface)**
- Surface minimale : 9 m². Surface maximale : 500 m².
- Les coordonnées sont extraites en découpant `_geopoint` au niveau de la virgule.
## 11. Coordonnées et neufs classés D, E ou G
On vérifie que les coordonnées sont convertibles et situées en France, puis on regarde les logements neufs avec une mauvaise étiquette.

In [ ]:
for d, df in dfs.items():
    coords = df["_geopoint"].str.split(",", expand=True)
    lat = pd.to_numeric(coords[0], errors="coerce")
    lon = pd.to_numeric(coords[1], errors="coerce")

    print(f"===== Département {d} =====")
    print("Non convertibles :", lat.isna().sum())
    print("Latitude  :", lat.min().round(2), "->", lat.max().round(2))
    print("Longitude :", lon.min().round(2), "->", lon.max().round(2))
    print("Hors de la France métropolitaine :",
          (~(lat.between(41, 52) & lon.between(-5.5, 10))).sum())
    print()

In [ ]:
cols = ["etiquette_dpe", "etiquette_ges", "conso_5_usages_par_m2_ep",
        "emission_ges_5_usages_par_m2", "surface_habitable_logement",
        "type_energie_principale_chauffage"]

for d, df in dfs.items():
    neufs_ge = df[(df["origine"] == "dpe_neufs") & df["etiquette_dpe"].isin(["E", "G"])]
    print(f"===== Département {d} : {len(neufs_ge)} neufs en E ou G =====")
    if len(neufs_ge):
        print(neufs_ge[cols].sort_values("etiquette_dpe"))
    print()

**Observations (étape 11)**
- Coordonnées : 22 points invalides à Lille et 184 à Lyon, tous situés à la même position (latitude -5,98, longitude -1,36), en plein océan. Probablement une valeur par défaut après un échec de géocodage. Paris est cohérent.
- Neufs en G à Lyon (12 lignes) : consommation de 58 à 83 kWh/m²/an (niveau A ou B) mais émissions de GES de 5 900 à 8 600 kg CO₂/m²/an, ce qui est impossible. L'étiquette G vient donc d'une erreur de saisie des émissions. Tous sont chauffés par un réseau de chauffage urbain.
- Neuf en G à Lille (1 ligne) et neufs en E (3 lignes) : étiquettes cohérentes avec la consommation, on les garde.

**Décisions (étape 11)**
- Coordonnées hors de la France métropolitaine : latitude et longitude remplacées par « manquant » (la ligne est conservée).
- Émissions de GES aberrantes : lignes supprimées (seuil à fixer).

## 12. Vérifications : coordonnées invalides et émissions de GES
On vérifie que les coordonnées invalides sont bien un échec de géocodage, puis on fixe le seuil des émissions de GES aberrantes.

In [ ]:
for d, df in dfs.items():
    coords = df["_geopoint"].str.split(",", expand=True)
    lat = pd.to_numeric(coords[0], errors="coerce")
    lon = pd.to_numeric(coords[1], errors="coerce")
    invalide = ~(lat.between(41, 52) & lon.between(-5.5, 10))

    print(f"===== Département {d} : {invalide.sum()} coordonnées invalides =====")
    print(df.loc[invalide, "_geopoint"].value_counts().head(3))
    print("Statut de géocodage :", df.loc[invalide, "statut_geocodage"].value_counts(dropna=False).to_dict())
    print("Type de logement    :", df.loc[invalide, "origine"].value_counts().to_dict())
    print()

In [ ]:
e = "emission_ges_5_usages_par_m2"

for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df[e].quantile([0.5, 0.99, 0.999, 1]).round(1).to_dict())
    print("Émissions > 200 :", (df[e] > 200).sum())
    print(df[df[e] > 200].groupby(["origine", "etiquette_dpe"]).size())
    print()

**Observations (étape 12)**
- Coordonnées invalides : 22 lignes à Lille et 184 à Lyon, toutes des logements existants, avec exactement la même position (latitude -5,98 ; longitude -1,36). Le statut de géocodage indique pourtant « adresse géocodée », donc l'erreur vient des données de l'ADEME et pas d'un échec visible. Paris n'est pas concerné.
- Émissions de GES : médiane d'environ 13 kg CO₂/m²/an et 99,9 % des logements sous 175. Les maximums plausibles sont de 456 (Lille) et 320 (Paris).
- À Lyon, les 12 neufs classés G ont des émissions de 5 900 à 8 600 : valeurs impossibles qui produisent une étiquette fausse.

**Décisions (étape 12)**
- Coordonnées hors de la France métropolitaine : latitude et longitude remplacées par « manquant ». La ligne est conservée et ne s'affiche pas sur la carte.
- Émissions de GES supérieures à 1 000 kg CO₂/m²/an : lignes supprimées (environ 12 lignes à Lyon). Cette vérification devra être refaite après application des règles.

## 13. Choix des colonnes : méthode
Sur les 224 colonnes, on ne garde que celles utiles à l'application.

**Critères de conservation** : une colonne est gardée si elle sert à identifier un DPE, à filtrer, à calculer un KPI ou un graphique, à placer un point sur la carte, ou à alimenter un modèle.

**Critères de suppression** : colonne technique (identifiants internes de l'API), trop détaillée ou trop vide (équipements secondaires, copropriété), résultat intermédiaire du calcul du DPE (déperditions, besoins, apports), ou adresse précise (inutile : la carte utilise le point géocodé).

**Cas particulier** : les colonnes qui contiennent déjà la réponse (consommations par usage, émissions de GES, coût) sont gardées pour l'analyse mais **exclues des modèles**.

**Limites** : le sens de chaque colonne est déduit de son nom. À confirmer dans le dictionnaire de données de l'ADEME. Les colonnes que les règles ne couvrent pas sont listées « à examiner » et décidées en équipe.

Le tableau complet (224 colonnes, décision et raison) est enregistré dans `docs/colonnes_retenues.csv`.

Une seule cellule applique toutes les décisions sur les 224 colonnes. Pour chaque colonne, la décision est prise dans cet ordre :
1. **Décision manuelle** (colonne par colonne, après examen des valeurs manquantes et des valeurs présentes).
2. **Règles générales** (par rôle ou par motif dans le nom de la colonne).
3. Sinon, la colonne est marquée « à examiner ».

Résultat : `docs/colonnes_retenues.csv` (colonne, groupe, décision, raison, taux de valeurs manquantes) et deux listes, `COLONNES_GARDEES` et `COLONNES_ANALYSE`.

In [ ]:
import os
import re

toutes = sorted(set().union(*[set(df.columns) for df in dfs.values()]))

# ---------------------------------------------------------------
# 1. Colonnes gardées, par rôle
# ---------------------------------------------------------------
IDENT = {"numero_dpe", "date_reception_dpe", "origine"}
CIBLES = {"etiquette_dpe", "conso_5_usages_par_m2_ep"}
GEO = {"code_departement_ban", "code_postal_ban", "nom_commune_ban", "code_insee_ban", "_geopoint"}
LOGEMENT = {"surface_habitable_logement", "periode_construction", "type_batiment",
            "nombre_niveau_logement", "hauteur_sous_plafond"}
CHAUFFAGE = {"type_energie_principale_chauffage", "type_installation_chauffage",
             "type_energie_principale_ecs", "type_installation_ecs"}
ISOLATION = {"qualite_isolation_enveloppe", "qualite_isolation_murs",
             "qualite_isolation_menuiseries", "qualite_isolation_plancher_bas"}
# Analyse seulement : contiennent déjà la réponse ou sont des résultats de calcul
ANALYSE = {"etiquette_ges", "emission_ges_5_usages_par_m2", "conso_5_usages_ep",
           "conso_chauffage_ep", "conso_ecs_ep", "conso_eclairage_ep",
           "conso_auxiliaires_ep", "cout_total_5_usages", "ubat_w_par_m2_k"}

# ---------------------------------------------------------------
# 2. Décisions manuelles : colonnes supprimées après examen
# ---------------------------------------------------------------
SUPPRESSIONS = [
    (["annee_construction"], "Redondante",
     "remplacée par periode_construction, complète"),
    (["zone_climatique", "classe_altitude"], "Redondante",
     "manquantes à 45-71 % chez les neufs ; le département porte l'information"),
    (["categorie_enr", "electricite_pv_autoconsommee", "nombre_module", "presence_production_pv",
      "surface_totale_capteurs_pv", "systeme_production_electricite_origine_renouvelable"],
     "Trop incomplète", "panneaux solaires et énergies renouvelables : manquantes à 65 % ou plus"),
    (["surface_climatisee", "type_energie_climatisation"],
     "Trop incomplète", "climatisation : manquantes à 98 % ou plus"),
    (["type_ventilation", "surface_ventilee"],
     "Trop incomplète", "ventilation : manquantes à 95 % ou plus"),
    (["indicateur_confort_ete", "isolation_toiture", "logement_traversant",
      "presence_brasseur_air", "protection_solaire_exterieure"],
     "Trop incomplète", "manquantes à 46 % chez les existants et à 90 % ou plus chez les neufs"),
    (["classe_inertie_batiment", "inertie_lourde"],
     "Trop incomplète", "manquantes à 96 % ou plus chez les neufs"),
    (["nombre_appartement", "nombre_niveau_immeuble", "numero_etage_appartement",
      "position_logement_dans_immeuble", "typologie_logement", "appartement_non_visite"],
     "Immeuble / diagnostic", "information sur l'immeuble ou le diagnostic, manquante à plus de 50 % pour un type"),
    (["qualite_isolation_plancher_haut_comble_amenage", "qualite_isolation_plancher_haut_comble_perdu",
      "qualite_isolation_plancher_haut_toit_terrasse"],
     "Trop incomplète", "isolation du plancher haut : manquantes à 54 % ou plus"),
    (["date_derniere_modification_dpe", "date_etablissement_dpe",
      "date_fin_validite_dpe", "date_visite_diagnostiqueur"],
     "Redondante", "date_reception_dpe suffit pour le suivi et le rafraîchissement"),
    (["coordonnee_cartographique_x_ban", "coordonnee_cartographique_y_ban"],
     "Redondante", "la carte utilise _geopoint (latitude et longitude)"),
    (["version_dpe"], "Redondante", "reflète la date de réception ; inconnue du particulier"),
    (["modele_dpe"], "Quasi constante",
     "une seule valeur chez les existants, plus de 96 % de RT2012 chez les neufs"),
    (["methode_application_dpe"], "Redondante", "recoupe origine et type_batiment"),
    (["ventilation_posterieure_2012"], "Quasi constante", "96 % à 99 % de 0"),
    (["production_electricite_pv_kwhep_par_an"], "Trop incomplète",
     "94 % à 96 % de zéros, valeurs aberrantes"),
    (["nombre_niveau_logement"], "Incohérente",
     "plus de 3 niveaux pour 29 % des appartements neufs, contre 2-3 % chez les existants ; manquante à 14-45 % chez les neufs"),
    (["type_energie_principale_ecs"], "Redondante",
     "« Non affecté » pour 100 % des neufs : recoupe origine"),
]
DECISIONS_MANUELLES = {c: (groupe, "supprimée", raison)
                       for cols, groupe, raison in SUPPRESSIONS for c in cols}

# ---------------------------------------------------------------
# 3. Règles générales (la première qui correspond décide)
# ---------------------------------------------------------------
REGLES = [
    ("Technique", "supprimée", lambda c: c in {"_id", "_i", "_rand", "_score"},
     "identifiant interne de l'API, sans information métier"),
    ("Identification", "gardée", lambda c: c in IDENT, "suivi des DPE, filtre neuf/existant, rafraîchissement"),
    ("Cible", "gardée", lambda c: c in CIBLES, "variable à analyser et à prédire"),
    ("Géographie", "gardée", lambda c: c in GEO, "carte et filtres géographiques"),
    ("Logement", "gardée", lambda c: c in LOGEMENT, "filtres, graphiques et modèles"),
    ("Chauffage / eau chaude", "gardée", lambda c: c in CHAUFFAGE, "filtres, graphiques et modèles"),
    ("Isolation", "gardée", lambda c: c in ISOLATION, "analyse, modèles à décider"),
    ("Analyse seulement", "gardée (hors modèles)", lambda c: c in ANALYSE,
     "contient déjà la réponse ou résultat de calcul : KPI et graphiques, pas les modèles"),
    ("Adresse détaillée", "supprimée",
     lambda c: bool(re.search(r"adresse|nom_rue|numero_voie|complement_adresse|identifiant_ban|score_ban|statut_geocodage|id_rnb|code_region_ban|_brut$", c)),
     "adresse précise, inutile : la carte utilise le point géocodé"),
    ("Résultat de calcul", "supprimée",
     lambda c: bool(re.match(r"(conso_|emission_ges_|cout_|deperditions_|besoin_|apport_)", c)),
     "résultat intermédiaire du calcul du DPE, détail non utile à l'app"),
    ("Équipement détaillé", "supprimée",
     lambda c: bool(re.search(r"_n[123]\b|_n[123]_|_installation_|generateur|description_", c)),
     "détail technique des équipements, très incomplet"),
    ("Copropriété / immeuble", "supprimée",
     lambda c: bool(re.search(r"copropriete|rpls|dpe_immeuble|dpe_remplace|nom_residence|surface_.*immeuble|surface_tertiaire", c)),
     "information sur le bâtiment ou la copropriété, hors sujet"),
]

def classer(col):
    """Renvoie (groupe, décision, raison) : décision manuelle d'abord, puis règles générales."""
    if col in DECISIONS_MANUELLES:
        return DECISIONS_MANUELLES[col]
    for groupe, decision, test, raison in REGLES:
        if test(col):
            return groupe, decision, raison
    return "À examiner", "à examiner", "non couverte par les règles : à décider en équipe"

# ---------------------------------------------------------------
# 4. Tableau final : décision, raison, taux de valeurs manquantes
# ---------------------------------------------------------------
manq = {}
for origine in ["dpe_existants", "dpe_neufs"]:
    par_dept = []
    for d, df in dfs.items():
        g = df[(df["type_batiment"] != "immeuble") & (df["origine"] == origine)]
        par_dept.append(g.reindex(columns=toutes).isna().mean() * 100)
    manq[origine] = pd.concat(par_dept, axis=1).max(axis=1).round(0)

tableau = pd.DataFrame([(c, *classer(c)) for c in toutes],
                       columns=["colonne", "groupe", "decision", "raison"])
tableau["manq_existants_%"] = tableau["colonne"].map(manq["dpe_existants"])
tableau["manq_neufs_%"] = tableau["colonne"].map(manq["dpe_neufs"])

print(tableau["decision"].value_counts(), "\n")
print("Encore à examiner :", tableau.loc[tableau["decision"] == "à examiner", "colonne"].tolist())

COLONNES_GARDEES = tableau.loc[tableau["decision"].str.startswith("gardée"), "colonne"].tolist()
COLONNES_ANALYSE = tableau.loc[tableau["decision"] == "gardée (hors modèles)", "colonne"].tolist()
print(len(COLONNES_GARDEES), "colonnes gardées, dont", len(COLONNES_ANALYSE), "hors modèles")

os.makedirs("../docs", exist_ok=True)
#tableau.to_csv("../docs/colonnes_retenues.csv", index=False, encoding="utf-8-sig")

**Bilan du choix des colonnes**
Sur 224 colonnes, **30 sont conservées** et 194 supprimées. Le détail (colonne, groupe, décision, raison, taux de valeurs manquantes) est dans `docs/colonnes_retenues.csv`.

| Groupe | Nombre | Usage |
|---|---|---|
| Identification | 3 | suivi, filtre neuf/existant |
| Cibles | 2 | variables à analyser et à prédire |
| Géographie | 5 | carte et filtres |
| Logement | 5 | filtres, graphiques, modèles |
| Chauffage et eau chaude | 4 | filtres, graphiques, modèles |
| Isolation | 4 | analyse, modèles à décider |
| Analyse seulement | 9 | KPI et graphiques, **exclues des modèles** |

Les colonnes d'analyse contiennent déjà la réponse (consommations, émissions de GES, coût) ou sont des résultats de calcul du DPE (`ubat_w_par_m2_k`). Elles seront listées dans `COLONNES_ANALYSE` pour être écartées des variables d'entrée des modèles.

## 14. Harmonisation des catégories
Pour chaque colonne de catégories conservée, on vérifie les valeurs présentes dans les trois départements : orthographe, casse, espaces, catégories rares ou incohérentes.

In [ ]:
categorielles = [
    "periode_construction", "type_batiment",
    "type_energie_principale_chauffage", "type_energie_principale_ecs",
    "type_installation_chauffage", "type_installation_ecs",
    "qualite_isolation_enveloppe", "qualite_isolation_murs",
    "qualite_isolation_menuiseries", "qualite_isolation_plancher_bas",
    "nombre_niveau_logement",
]

for col in categorielles:
    valeurs = pd.concat([df[col] for df in dfs.values()]).dropna().astype(str)
    print(f"===== {col} : {valeurs.nunique()} valeurs distinctes =====")

    tab = pd.DataFrame({d: df[col].value_counts(dropna=False) for d, df in dfs.items()})
    print(tab.fillna(0).astype(int).sort_values("59", ascending=False).head(12))

    # Même valeur écrite de plusieurs façons (casse ou espaces) ?
    if valeurs.str.strip().str.lower().nunique() < valeurs.nunique():
        print("Une même valeur est écrite de plusieurs façons (casse ou espaces)")
    print()

**Observations (étape 14)**
- Aucune valeur écrite de plusieurs façons (casse, espaces) : pas de correction nécessaire. Un `strip()` sera tout de même appliqué dans le prétraitement pour les données futures de l'API.
- `periode_construction` : 10 périodes cohérentes entre les trois départements. Un ordre chronologique sera défini pour les filtres.
- `type_batiment` : 3 valeurs (appartement, maison, immeuble). Les immeubles (1 271 lignes) sont exclus.
- `type_energie_principale_chauffage` : 13 valeurs, dont plusieurs très rares (charbon : 6 lignes, GPL : 39, électricité d'origine renouvelable : 16).
- `type_installation_chauffage` et `type_installation_ecs` sont manquantes sur les mêmes lignes.
- `qualite_isolation_*` : 4 niveaux ordonnés (insuffisante, moyenne, bonne, très bonne).
- `nombre_niveau_logement` : valeurs jusqu'à 16 niveaux, peu plausibles pour un seul logement. À vérifier.
- `type_energie_principale_ecs` : la valeur « Non affecté » est la plus fréquente (environ 70 % des lignes à Lille et à Lyon, 48 % à Paris). Son sens est à vérifier.

**Décisions provisoires (étape 14)**
- Énergie de chauffage : les 4 variantes « Bois – … » sont regroupées en « Bois » ; GPL, charbon, propane et électricité d'origine renouvelable en « Autre ». Les valeurs inconnues futures vont dans « Autre ».
- `type_installation_chauffage` et `type_installation_ecs` : manquants remplacés par « inconnu ».
- `qualite_isolation_*` : conservées avec leur ordre ; manquants traités dans le pipeline des modèles.
- `nombre_niveau_logement` et `type_energie_principale_ecs` : à décider après vérification.

In [ ]:
# 1. Toutes les valeurs de l'énergie de chauffage (le tableau précédent s'arrêtait à 12)
print(pd.concat([df["type_energie_principale_chauffage"] for df in dfs.values()]).value_counts(), "\n")

for d, df in dfs.items():
    logements = df[df["type_batiment"] != "immeuble"]
    niv = logements["nombre_niveau_logement"]
    print(f"===== Département {d} (hors immeubles) =====")

    # 2. Part des logements avec plus de 3 niveaux, ou sans valeur, par type
    print(logements.assign(plus_de_3_niveaux=niv > 3, niveaux_manquants=niv.isna())
                   .groupby(["origine", "type_batiment"])[["plus_de_3_niveaux", "niveaux_manquants"]]
                   .mean().round(2))

    # 3. Installation de chauffage manquante, par type
    print(logements.assign(installation_manquante=logements["type_installation_chauffage"].isna())
                   .groupby(["origine", "type_batiment"])["installation_manquante"].mean().round(2))

    # 4. Eau chaude « Non affecté », par type de logement
    print(logements.assign(ecs_non_affecte=logements["type_energie_principale_ecs"].str.startswith("Non aff", na=False))
                   .groupby("origine")["ecs_non_affecte"].mean().round(2))
    print()

In [ ]:
# Ordres des catégories ordonnées
ORDRE_PERIODE = ["avant 1948", "1948-1974", "1975-1977", "1978-1982", "1983-1988",
                 "1989-2000", "2001-2005", "2006-2012", "2013-2021", "après 2021"]
ORDRE_ISOLATION = ["insuffisante", "moyenne", "bonne", "très bonne"]

def regrouper_energie(valeur):
    """Regroupe les énergies de chauffage rares ou proches."""
    if pd.isna(valeur):
        return "Inconnu"
    v = str(valeur).strip()
    if v.startswith("Bois"):
        return "Bois"
    if v in {"Électricité", "Gaz naturel", "Réseau de Chauffage urbain",
             "Fioul domestique", "Non affecté"}:
        return v
    return "Autre"

# Contrôle sur les trois départements (sans modifier les données)
for d, df in dfs.items():
    print(f"===== Département {d} =====")
    print(df["type_energie_principale_chauffage"].map(regrouper_energie).value_counts().to_dict())
    print("Périodes hors liste :", set(df["periode_construction"].dropna()) - set(ORDRE_PERIODE))
    print("Installation chauffage :", df["type_installation_chauffage"].fillna("non applicable").value_counts().to_dict())
    print()

## 15. Application du prétraitement
Les règles décidées dans ce notebook sont écrites dans `src/preprocessing.py`. On les applique à chaque département séparément (mêmes règles) et on affiche le bilan : nombre de lignes après chaque étape.

In [ ]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2
from src.preprocessing import DataPreprocessor, COLONNES_GARDEES

RAW = "../data/raw/2026-10-12_132605/"
datasets = {
    d: {"dpe_existants": pd.read_parquet(RAW + f"{d}/dpe_existants.parquet"),
        "dpe_neufs": pd.read_parquet(RAW + f"{d}/dpe_neufs.parquet")}
    for d in DEPARTEMENTS
}

prepro = DataPreprocessor()
propres = prepro.process(datasets)

print({d: df.shape for d, df in propres.items()})
pd.concat([b.rename(columns={"lignes": d}) for d, b in prepro.bilans.items()], axis=1)

In [ ]:
for d, df in propres.items():
    print(f"===== {d} =====")
    print(pd.crosstab(df["origine"], df["etiquette_dpe"]))
    print(df[["surface_habitable_logement", "conso_5_usages_par_m2_ep",
              "emission_ges_5_usages_par_m2"]].describe().round(1).loc[["min", "50%", "max"]])
    print("Coordonnées manquantes :", df["latitude"].isna().sum(),
          "| hauteur manquante :", df["hauteur_sous_plafond"].isna().sum())
    print()

In [ ]:
for d, etiq in [("59", "G"), ("75", "E")]:
    n = datasets[d]["dpe_neufs"]
    n = n[n["etiquette_dpe"] == etiq]
    print(f"===== {d}, neufs en {etiq} =====")
    print(n[["numero_dpe", "type_batiment", "surface_habitable_logement",
             "conso_5_usages_par_m2_ep", "emission_ges_5_usages_par_m2"]])

In [ ]:
tous = pd.concat(propres.values(), keys=propres.keys(), names=["dept"])

# 1. Valeurs manquantes qui restent, en %
manq = (tous.isna().mean() * 100).round(1)
print(manq[manq > 0].sort_values(ascending=False), "\n")

# 2. Étiquette GES : valeurs présentes
print(tous["etiquette_ges"].value_counts(dropna=False).sort_index(), "\n")

# 3. Valeurs extrêmes des colonnes d'analyse
cols = ["conso_5_usages_ep", "conso_chauffage_ep", "conso_ecs_ep", "conso_eclairage_ep",
        "conso_auxiliaires_ep", "cout_total_5_usages", "ubat_w_par_m2_k"]
print(tous[cols].describe(percentiles=[0.5, 0.99, 0.999]).round(1).T[["min", "50%", "99%", "99.9%", "max"]], "\n")

# 4. Cohérence (hypothèse : conso_5_usages_ep est la consommation totale annuelle)
ecart = (tous["conso_5_usages_ep"] / tous["surface_habitable_logement"]
         - tous["conso_5_usages_par_m2_ep"]).abs()
print("Lignes où l'écart dépasse 5 kWh/m²/an :", int((ecart > 5).sum()))

In [ ]:
# 1. Les 128 lignes où conso_5_usages_ep / surface ne retombe pas sur la cible
tous["ecart"] = ecart
bizarres = tous[ecart > 5]
print(bizarres.groupby(["dept", "origine"]).size(), "\n")
print(bizarres[["surface_habitable_logement", "conso_5_usages_par_m2_ep",
                "conso_5_usages_ep", "ecart"]].describe().round(1).loc[["min", "50%", "max"]], "\n")

# 2. Part des auxiliaires dans la consommation totale
part_aux = tous["conso_auxiliaires_ep"] / tous["conso_5_usages_ep"]
print(part_aux.describe(percentiles=[0.5, 0.99, 0.999]).round(2), "\n")
tous.loc[part_aux > 0.5, ["origine", "surface_habitable_logement", "etiquette_dpe",
                          "conso_5_usages_par_m2_ep", "conso_auxiliaires_ep"]].head(8)

In [ ]:
usages = ["conso_chauffage_ep", "conso_ecs_ep", "conso_eclairage_ep", "conso_auxiliaires_ep"]
somme = tous[usages].sum(axis=1)

# Le total des usages ne peut pas dépasser la consommation totale (le refroidissement n'est pas gardé)
incoh = somme > tous["conso_5_usages_ep"] * 1.01
part_aux = tous["conso_auxiliaires_ep"] / tous["conso_5_usages_ep"]

print("Somme des usages > total :", int(incoh.sum()), "lignes")
print("Auxiliaires > total      :", int((part_aux > 1).sum()), "lignes\n")
print(tous[incoh].groupby(["dept", "origine"]).size(), "\n")
print(tous.loc[part_aux > 1, ["origine", "conso_5_usages_ep", "conso_auxiliaires_ep"]].head(8))

In [ ]:
bilan = pd.concat({d: b["lignes"] for d, b in prepro.bilans.items()}, axis=1)
bilan["total"] = bilan.sum(axis=1)
bilan.to_csv("../docs/bilan_nettoyage.csv", encoding="utf-8-sig")
bilan

## 16. Conclusion du nettoyage

### 16.1 Données de départ
- 3 départements (59, 69, 75), 58 644 lignes, 224 colonnes (221 pour Paris).
- Logements neufs et existants fusionnés par département, distingués par la colonne `origine`.
- Aucun doublon sur `numero_dpe`.

### 16.2 Limites des données (à reprendre dans le rapport ML)
- **Échantillon plafonné** : 10 000 lignes par type et par département (limite d'une requête API). Paris compte 8 644 neufs : on a probablement tous les neufs de Paris.
- **Pas un tirage aléatoire** : les existants de Lille et de Lyon sont concentrés sur 2021 et 2026, ceux de Paris sur 2026. Lille et Lyon s'arrêtent au 09/02/2026. On ne peut pas étudier l'évolution dans le temps.
- **La proportion neufs / existants (environ 50/50) est un artefact** du plafond, pas la composition du parc.
- **Étiquettes déséquilibrées** : le C domine, F et G sont rares, les neufs sont presque tous en A, B ou C.
- **Les départements diffèrent** : Paris a davantage de E, F et G et des consommations plus élevées (parc ancien, petites surfaces). C'est une vraie différence, pas une erreur.
- **Le sens de chaque colonne est déduit de son nom** : à confirmer dans le dictionnaire de données de l'ADEME.

### 16.3 Règles qui suppriment des lignes
| Règle | Seuil | Justification | Lignes concernées (ordre de grandeur) |
|---|---|---|---|
| Doublons | `numero_dpe` | éviter de compter deux fois un DPE | 0 |
| Immeubles exclus | `type_batiment` = immeuble | un DPE d'immeuble décrit un bâtiment entier (surface médiane d'environ 1 000 m²) ; l'application concerne un logement | 1 271 |
| Surface manquante | ligne supprimée | variable indispensable | environ 317 (hors immeubles) |
| Surface minimale | 9 m² | en dessous, ce n'est pas un logement ; 90 cas sur 95 sont à Paris | environ 95 |
| Surface maximale | 500 m² | de 200 à 500 m², grandes maisons plausibles ; au-delà, erreurs de saisie probables | environ 17 |
| Étiquette | A à G | étiquettes valides | aucune hors liste constatée |
| Consommation | supérieure à 0 et au plus 1 000 kWh/m²/an | la classe G commence vers 420 ; un seuil à 600 supprimerait de vraies passoires thermiques ; au-dessus de 1 000, quelques valeurs isolées | au plus 18 |
| Émissions de GES | au plus 1 000 kg CO₂/m²/an | les valeurs plausibles vont jusqu'à environ 460 ; les 12 neufs de Lyon classés G ont des émissions de 5 900 à 8 600 : étiquette fausse | environ 12 |

- **Colonnes de consommation par usage non additives** : la somme des 4 usages (chauffage, eau chaude, éclairage, auxiliaires) dépasse la consommation totale pour 29 164 lignes (51 % ; environ 69 % des neufs et 34 % des existants), et les auxiliaires dépassent le total pour 131 lignes. La proportion est trop élevée pour des erreurs de saisie : c'est probablement une question de définition des colonnes, à vérifier dans le dictionnaire de données de l'ADEME.
- **Décision** : aucune ligne supprimée. La cible `conso_5_usages_par_m2_ep` est valide. Les colonnes par usage ne sont ni additionnées ni exprimées en parts : les KPI et graphiques utilisent la consommation par m².

### 16.4 Règles qui conservent la ligne
| Règle | Justification | Lignes concernées |
|---|---|---|
| Coordonnées hors de la France métropolitaine → manquantes | 206 logements existants (22 à Lille, 184 à Lyon) partagent la même position erronée, en plein océan | 206 |
| Hauteur sous plafond hors de 1,8 à 5 m → manquante | unité de saisie erronée probable (270 ou 310 en cm, 25 ou 26 multipliées par 10) | environ 130 |
| Énergies de chauffage regroupées | « Bois » (4 variantes, 863 lignes), « Autre » (GPL, propane, charbon, électricité d'origine renouvelable : 62 lignes) | toutes |
| Installation de chauffage et d'eau chaude manquante → « non applicable » | colonne vide pour 100 % des maisons, renseignée pour les appartements | maisons |
| Espaces supprimés dans les catégories | protection pour les futures données de l'API | toutes |
- **Exclusion des immeubles** : un DPE classé « immeuble » peut parfois décrire un logement isolé (par exemple un neuf de Lille classé G et un neuf de Paris classé E sont dans ce cas). Ils sont exclus avec les 1 271 lignes d'immeubles ; la perte est faible (2,2 %) et la règle protège l'application des bâtiments entiers.

### 16.5 Colonnes : 224 → 30
| Groupe | Colonnes | Usage |
|---|---|---|
| Identification (3) | `numero_dpe`, `date_reception_dpe`, `origine` | suivi, filtre neuf / existant, rafraîchissement |
| Cibles (2) | `etiquette_dpe`, `conso_5_usages_par_m2_ep` | variables à analyser et à prédire |
| Géographie (5) | département, code postal, commune, code INSEE, `_geopoint` | carte et filtres |
| Logement (3) | surface, période de construction, type de bâtiment | filtres, graphiques, modèles |
| Hauteur sous plafond (1) | `hauteur_sous_plafond` | à décider pour les modèles |
| Chauffage et eau chaude (3) | énergie de chauffage, type d'installation de chauffage et d'eau chaude | filtres, graphiques, modèles |
| Isolation (4) | enveloppe, murs, menuiseries, plancher bas | analyse, modèles à décider |
| Analyse seulement (9) | étiquette GES, émissions de GES, consommations par usage, coût, `ubat_w_par_m2_k` | KPI et graphiques, **exclues des modèles** |

194 colonnes supprimées : techniques, adresses détaillées, résultats intermédiaires du calcul du DPE, équipements détaillés, colonnes trop incomplètes (de 46 % à 100 % de manquants pour au moins un type de logement), redondantes (`annee_construction`, `version_dpe`, dates secondaires) ou incohérentes (`nombre_niveau_logement`, `type_energie_principale_ecs`). Le détail, colonne par colonne, est dans `docs/colonnes_retenues.csv`.

### 16.6 Ce qui n'est pas décidé ici : étape de modélisation
- **Choix des variables d'entrée** : les colonnes d'analyse sont exclues (elles contiennent déjà la réponse). Pour les autres, à décider : `hauteur_sous_plafond`, les qualités d'isolation (un particulier ne les connaît pas forcément), `origine`, le département.
- **Valeurs manquantes** : imputées dans le pipeline scikit-learn, sur le jeu d'entraînement seulement (hauteur, coordonnées, quelques isolations).
- **Encodage** : catégories ordonnées (période de construction, qualités d'isolation) et non ordonnées.
- **Déséquilibre des classes** : découpage stratifié, pondération des classes, métriques F1 macro et matrice de confusion.
- **Évaluation séparée** : à la fois sur l'ensemble et sur les logements existants seuls, car les neufs gonflent les performances.
- **Sensibilité des seuils** : refaire les modèles avec un seuil de consommation de 800 pour vérifier que les résultats ne changent pas.

### 16.7 Points à confirmer
- Le sens des colonnes dans le dictionnaire de données de l'ADEME.
- Auprès de l'équipe : l'extraction pourrait-elle utiliser un tirage aléatoire (colonne `_rand`) plutôt que les premiers résultats ?
- Le nombre de lignes final (section 15) pour compléter le tableau 16.3.